# CViTS-Net Training Pipeline
## APTOS2019 Blindness Detection Dataset

Complete training notebook for CViTS-Net architecture with all metrics and visualizations.

## 1. Setup - Import Libraries & Configure Environment

In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add current directory to path
sys.path.insert(0, os.getcwd())

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import time
import traceback
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Memory: {props.total_memory / 1024**3:.1f} GB")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set seeds
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("\n✓ Environment configured successfully")

PyTorch version: 2.9.1+cu126
CUDA Available: True
GPU: NVIDIA RTX A5000
GPU Memory: 22.5 GB
Using device: cuda

✓ Environment configured successfully


## 2. Import Custom Modules

In [2]:
from dataset_loader import get_data_loaders
from cvitsnet_model import build_cvitsnet, count_parameters
from metrics import MetricsCalculator
from visualize import TrainingVisualizer

print("✓ All custom modules imported successfully")

✓ All custom modules imported successfully


## 3. Load Dataset

In [3]:
print("\n" + "="*80)
print("Loading APTOS2019 Dataset")
print("="*80)

# Load data with retry logic
max_retries = 3
for attempt in range(max_retries):
    try:
        print(f"\nAttempt {attempt + 1}/{max_retries}...")
        
        train_dataset, val_dataset, test_dataset, class_weights, \
        (X_train, y_train, X_val, y_val, X_test, y_test) = \
        get_data_loaders(dataset_path="APTOS2019", batch_size=16)
        
        print("✓ Dataset loaded successfully!")
        print(f"  Train samples: {len(X_train)}")
        print(f"  Val samples: {len(X_val)}")
        print(f"  Test samples: {len(X_test)}")
        break
        
    except Exception as e:
        print(f"Error: {str(e)}")
        if attempt < max_retries - 1:
            wait_time = 2 ** attempt
            print(f"Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
        else:
            raise


Loading APTOS2019 Dataset

Attempt 1/3...
Total training samples: 3662
Class distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64
Loaded 100/3662 images
Loaded 200/3662 images
Loaded 300/3662 images
Loaded 400/3662 images
Loaded 500/3662 images
Loaded 600/3662 images
Loaded 700/3662 images
Loaded 800/3662 images
Loaded 900/3662 images
Loaded 1000/3662 images
Loaded 1100/3662 images
Loaded 1200/3662 images
Loaded 1300/3662 images
Loaded 1400/3662 images
Loaded 1500/3662 images
Loaded 1600/3662 images
Loaded 1700/3662 images
Loaded 1800/3662 images
Loaded 1900/3662 images
Loaded 2000/3662 images
Loaded 2100/3662 images
Loaded 2200/3662 images
Loaded 2300/3662 images
Loaded 2400/3662 images
Loaded 2500/3662 images
Loaded 2600/3662 images
Loaded 2700/3662 images
Loaded 2800/3662 images
Loaded 2900/3662 images
Loaded 3000/3662 images
Loaded 3100/3662 images
Loaded 3200/3662 images
Loaded 3300/3662 images
Loaded 3400/3662 images
Loaded 3500/366

## 4. Build CViTS-Net Model

In [4]:
print("\n" + "="*80)
print("Building CViTS-Net Model")
print("="*80)

# Build model
model = build_cvitsnet(num_classes=5, image_size=224)
model = model.to(device)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

total_params = count_parameters(model)
print(f"\nTotal Trainable Parameters: {total_params:,}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print("\n✓ Model built successfully!")


Building CViTS-Net Model

Total Trainable Parameters: 30,338,053
Device: cuda
GPU: NVIDIA RTX A5000

✓ Model built successfully!


## 5. Create Output Directories

In [5]:
# Create output directories
output_dir = Path("results")
model_dir = Path("trained_model")
plots_dir = output_dir / "Plots"
logs_dir = output_dir / "logs"

output_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

# Initialize visualizer
visualizer = TrainingVisualizer(str(plots_dir))

print(f"✓ Output directories created:")
print(f"  - {output_dir}")
print(f"  - {model_dir}")
print(f"  - {plots_dir}")
print(f"  - {logs_dir}")

✓ Output directories created:
  - results
  - trained_model
  - results\Plots
  - results\logs


## 6. Training Loop

In [6]:
print("\n" + "="*80)
print("Starting Training (100 epochs)")
print("="*80)

# Initialize metrics history
history = {
    'loss': {'train': [], 'val': []},
    'accuracy': {'train': [], 'val': []},
    'precision': {'train': [], 'val': []},
    'recall': {'train': [], 'val': []},
    'f1_score': {'train': [], 'val': []},
    'specificity': {'train': [], 'val': []},
    'roc_auc': {'train': [], 'val': []},
    'qwk': {'train': [], 'val': []}
}

epochs = 100
batch_size = 16

# Best model tracking (monitor val_qwk, mode=max)
best_val_qwk = -1.0
best_epoch = 0

# Training loop
for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    print("-" * 80)
    
    # Training phase
    model.train()
    train_metrics_calc = MetricsCalculator(num_classes=5)
    train_loss = 0.0
    num_batches = 0
    
    for batch_images, batch_labels in train_dataset:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        predictions = model(batch_images)
        loss = F.cross_entropy(predictions, batch_labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        train_loss += loss.item()
        pred_proba = F.softmax(predictions, dim=1).detach().cpu().numpy()
        train_metrics_calc.update(batch_labels.cpu().numpy(), pred_proba)
        num_batches += 1
    
    train_loss = train_loss / num_batches
    train_metrics = train_metrics_calc.compute_metrics()
    
    # Validation phase
    model.eval()
    val_metrics_calc = MetricsCalculator(num_classes=5)
    val_loss = 0.0
    num_val_batches = 0
    
    with torch.no_grad():
        for batch_images, batch_labels in val_dataset:
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.to(device)
            
            predictions = model(batch_images)
            loss = F.cross_entropy(predictions, batch_labels)
            
            val_loss += loss.item()
            pred_proba = F.softmax(predictions, dim=1).cpu().numpy()
            val_metrics_calc.update(batch_labels.cpu().numpy(), pred_proba)
            num_val_batches += 1
    
    val_loss = val_loss / num_val_batches
    val_metrics = val_metrics_calc.compute_metrics()
    
    # Store history
    history['loss']['train'].append(float(train_loss))
    history['loss']['val'].append(float(val_loss))
    for metric_name in ['accuracy', 'precision', 'recall', 'f1_score', 'specificity', 'roc_auc', 'qwk']:
        if metric_name in train_metrics:
            history[metric_name]['train'].append(train_metrics[metric_name])
            history[metric_name]['val'].append(val_metrics[metric_name])
    
    # Current val QWK
    current_val_qwk = val_metrics.get('qwk', 0)
    
    # Print epoch results
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"Train Acc: {train_metrics.get('accuracy', 0):.4f} | Val Acc: {val_metrics.get('accuracy', 0):.4f}")
    print(f"Train F1: {train_metrics.get('f1_score', 0):.4f} | Val F1: {val_metrics.get('f1_score', 0):.4f}")
    print(f"Train QWK: {train_metrics.get('qwk', 0):.4f} | Val QWK: {current_val_qwk:.4f}")
    
    # Save best model based on val_qwk (monitor="val_qwk", mode="max")
    if current_val_qwk > best_val_qwk:
        best_val_qwk = current_val_qwk
        best_epoch = epoch + 1
        best_model_path = model_dir / "best_cvitsnet_model.pth"
        torch.save({
            'epoch': best_epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_qwk': best_val_qwk,
            'val_loss': val_loss,
        }, str(best_model_path))
        print(f"✓ Best model saved (val_QWK: {best_val_qwk:.4f}, epoch: {best_epoch})")
    
    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        checkpoint_path = model_dir / f"checkpoint_epoch_{epoch + 1:03d}.pth"
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, str(checkpoint_path))
        print(f"Checkpoint saved: {checkpoint_path}")

print("\n" + "="*80)
print("Training completed!")
print(f"Best model: Epoch {best_epoch} (val_QWK: {best_val_qwk:.4f})")
print("="*80)


Starting Training (100 epochs)

Epoch 1/100
--------------------------------------------------------------------------------
Train Loss: 20.1780 | Val Loss: 8.6306
Train Acc: 0.3755 | Val Acc: 0.5087
Train F1: 0.3779 | Val F1: 0.4344
Train QWK: 0.0892 | Val QWK: 0.2902
✓ Best model saved (val_QWK: 0.2902, epoch: 1)

Epoch 2/100
--------------------------------------------------------------------------------
Train Loss: 3.9061 | Val Loss: 2.9829
Train Acc: 0.5287 | Val Acc: 0.4884
Train F1: 0.5240 | Val F1: 0.4534
Train QWK: 0.3767 | Val QWK: 0.3740
✓ Best model saved (val_QWK: 0.3740, epoch: 2)

Epoch 3/100
--------------------------------------------------------------------------------
Train Loss: 2.4572 | Val Loss: 1.8855
Train Acc: 0.5562 | Val Acc: 0.6773
Train F1: 0.5557 | Val F1: 0.6146
Train QWK: 0.4401 | Val QWK: 0.6594
✓ Best model saved (val_QWK: 0.6594, epoch: 3)

Epoch 4/100
--------------------------------------------------------------------------------
Train Loss: 1.7851

## 7. Test Set Evaluation

In [7]:
print("\n" + "="*80)
print("Evaluating on Test Set (using best model by val_QWK)")
print("="*80)

# Load best model (saved based on highest val_QWK)
best_model_path = model_dir / "best_cvitsnet_model.pth"
if best_model_path.exists():
    best_checkpoint = torch.load(str(best_model_path), map_location=device)
    model.load_state_dict(best_checkpoint['model_state_dict'])
    print(f"✓ Loaded best model from Epoch {best_checkpoint['epoch']} (val_QWK: {best_checkpoint['val_qwk']:.4f})")
else:
    print("⚠ Best model not found, using current model weights")
    best_checkpoint = {'epoch': epochs, 'val_qwk': history['qwk']['val'][-1], 'val_loss': history['loss']['val'][-1]}

model.eval()
test_metrics_calc = MetricsCalculator(num_classes=5)
test_loss = 0.0
num_test_batches = 0

with torch.no_grad():
    for batch_images, batch_labels in test_dataset:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)
        
        predictions = model(batch_images)
        loss = F.cross_entropy(predictions, batch_labels)
        
        test_loss += loss.item()
        pred_proba = F.softmax(predictions, dim=1).cpu().numpy()
        test_metrics_calc.update(batch_labels.cpu().numpy(), pred_proba)
        num_test_batches += 1

test_loss = test_loss / num_test_batches
test_metrics = test_metrics_calc.compute_metrics()

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_metrics.get('accuracy', 0):.4f}")
print(f"Test Precision: {test_metrics.get('precision', 0):.4f}")
print(f"Test Recall: {test_metrics.get('recall', 0):.4f}")
print(f"Test F1 Score: {test_metrics.get('f1_score', 0):.4f}")
print(f"Test Specificity: {test_metrics.get('specificity', 0):.4f}")
print(f"Test ROC-AUC: {test_metrics.get('roc_auc', 0):.4f}")
print(f"Test QWK: {test_metrics.get('qwk', 0):.4f}")

# Get confusion matrix and ROC curves
cm = test_metrics_calc.get_confusion_matrix()
roc_data = test_metrics_calc.get_roc_curves()

# Store test results
test_history = {
    'loss': float(test_loss),
    'metrics': test_metrics,
    'confusion_matrix': cm.tolist(),
    'roc_data': {k: {kk: v.tolist() for kk, v in vv.items()} for k, vv in roc_data.items()},
    'best_model_epoch': best_checkpoint['epoch'],
    'best_model_val_qwk': float(best_checkpoint['val_qwk'])
}

print(f"\n✓ Test evaluation complete (best model from epoch {best_checkpoint['epoch']})")


Evaluating on Test Set (using best model by val_QWK)
✓ Loaded best model from Epoch 91 (val_QWK: 0.7313)

Test Loss: 0.9972
Test Accuracy: 0.7249
Test Precision: 0.7217
Test Recall: 0.7249
Test F1 Score: 0.6929
Test Specificity: 0.9242
Test ROC-AUC: 0.9053
Test QWK: 0.7146

✓ Test evaluation complete (best model from epoch 91)


## 8. Save Model and Training History

In [8]:
print("\n" + "="*80)
print("Saving Model and Training History")
print("="*80)

# Save model (PyTorch format)
model_path = model_dir / "cvitsnet_aptos2019.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, str(model_path))
print(f"\n✓ Model saved: {model_path}")

# Save training history
def convert_types(obj):
    if isinstance(obj, dict):
        return {k: convert_types(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_types(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    else:
        return obj

history_data = {
    'training_history': convert_types(history),
    'test_results': convert_types(test_history)
}

history_path = logs_dir / "training_history.json"
with open(history_path, 'w') as f:
    json.dump(history_data, f, indent=4)
print(f"✓ Training history saved: {history_path}")

# Save best model metrics JSON (all 7 metrics from test evaluation with best model)
best_model_metrics = {
    'best_model_epoch': int(test_history.get('best_model_epoch', best_epoch)),
    'best_model_val_qwk': float(test_history.get('best_model_val_qwk', best_val_qwk)),
    'test_loss': float(test_history['loss']),
    'test_metrics': {
        'accuracy': float(test_metrics.get('accuracy', 0)),
        'precision': float(test_metrics.get('precision', 0)),
        'recall': float(test_metrics.get('recall', 0)),
        'f1_score': float(test_metrics.get('f1_score', 0)),
        'specificity': float(test_metrics.get('specificity', 0)),
        'roc_auc': float(test_metrics.get('roc_auc', 0)),
        'qwk': float(test_metrics.get('qwk', 0))
    }
}

best_metrics_path = logs_dir / "best_model_metrics.json"
with open(best_metrics_path, 'w') as f:
    json.dump(best_model_metrics, f, indent=4)
print(f"✓ Best model metrics saved: {best_metrics_path}")


Saving Model and Training History

✓ Model saved: trained_model\cvitsnet_aptos2019.pth
✓ Training history saved: results\logs\training_history.json
✓ Best model metrics saved: results\logs\best_model_metrics.json


## 9. Generate Visualizations

In [9]:
import matplotlib.pyplot as plt

print("\n" + "="*80)
print("Generating Visualizations")
print("="*80)

# Plot all metrics
visualizer.plot_all_metrics(history)

# Plot all metrics on one figure
visualizer.plot_metrics_comparison(history)

# Plot confusion matrix
class_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
visualizer.plot_confusion_matrix(cm, class_names)

# Plot ROC curves
if roc_data:
    fpr_dict = {}
    tpr_dict = {}
    roc_auc_dict = {}
    
    for i, (class_name, roc_vals) in enumerate(roc_data.items()):
        fpr_dict[i] = roc_vals['fpr']
        tpr_dict[i] = roc_vals['tpr']
        from sklearn.metrics import auc
        roc_auc_dict[i] = auc(roc_vals['fpr'], roc_vals['tpr'])
    
    visualizer.plot_roc_curve(fpr_dict, tpr_dict, roc_auc_dict)

# ============================================================
# QWK Curve Plot (Train & Validation)
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))
epochs_range = range(1, len(history['qwk']['train']) + 1)

ax.plot(epochs_range, history['qwk']['train'], 'b-', linewidth=2, label='Train QWK')
ax.plot(epochs_range, history['qwk']['val'], 'r-', linewidth=2, label='Validation QWK')

# Mark best validation QWK
best_val_qwk_idx = int(np.argmax(history['qwk']['val']))
best_val_qwk_value = history['qwk']['val'][best_val_qwk_idx]
ax.axvline(x=best_val_qwk_idx + 1, color='green', linestyle='--', alpha=0.7, label=f'Best Val QWK: {best_val_qwk_value:.4f} (Epoch {best_val_qwk_idx + 1})')
ax.scatter([best_val_qwk_idx + 1], [best_val_qwk_value], color='green', s=100, zorder=5)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Quadratic Weighted Kappa (QWK)', fontsize=12)
ax.set_title('CViTS-Net — QWK over Training Epochs', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(1, len(history['qwk']['train']))

plt.tight_layout()
qwk_plot_path = str(plots_dir / 'qwk_curve.png')
plt.savefig(qwk_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ QWK curve saved: {qwk_plot_path}")

# Save metrics JSON
visualizer.save_metrics_json(history_data, 'metrics.json')

print("\n✓ Visualizations generated successfully!")


Generating Visualizations
Saved: results\Plots\loss_vs_epoch.png
Saved: results\Plots\accuracy_vs_epoch.png
Saved: results\Plots\precision_vs_epoch.png
Saved: results\Plots\recall_vs_epoch.png
Saved: results\Plots\f1_score_vs_epoch.png
Saved: results\Plots\all_metrics.png
Saved: results\Plots\confusion_matrix.png
Saved: results\Plots\roc_curve.png
✓ QWK curve saved: results\Plots\qwk_curve.png
Saved: results\Plots\metrics.json

✓ Visualizations generated successfully!


## 10. Summary

In [10]:
print("\n" + "="*80)
print("TRAINING PIPELINE COMPLETED SUCCESSFULLY!")
print("="*80)

print(f"\n📊 Results Summary:")
print(f"\n  Final Training Metrics (Epoch {epochs}):")
print(f"    Loss: {history['loss']['train'][-1]:.4f}")
print(f"    Accuracy: {history['accuracy']['train'][-1]:.4f}")
print(f"    F1 Score: {history['f1_score']['train'][-1]:.4f}")
print(f"    ROC-AUC: {history['roc_auc']['train'][-1]:.4f}")
print(f"    QWK: {history['qwk']['train'][-1]:.4f}")

print(f"\n  Final Validation Metrics (Epoch {epochs}):")
print(f"    Loss: {history['loss']['val'][-1]:.4f}")
print(f"    Accuracy: {history['accuracy']['val'][-1]:.4f}")
print(f"    F1 Score: {history['f1_score']['val'][-1]:.4f}")
print(f"    ROC-AUC: {history['roc_auc']['val'][-1]:.4f}")
print(f"    QWK: {history['qwk']['val'][-1]:.4f}")

print(f"\n  Best Model: Epoch {best_model_metrics['best_model_epoch']} (val_QWK: {best_model_metrics['best_model_val_qwk']:.4f})")
print(f"\n  Test Set Metrics (Best Model):")
print(f"    Loss: {test_history['loss']:.4f}")
print(f"    Accuracy: {test_metrics['accuracy']:.4f}")
print(f"    Precision: {test_metrics['precision']:.4f}")
print(f"    Recall: {test_metrics['recall']:.4f}")
print(f"    F1 Score: {test_metrics['f1_score']:.4f}")
print(f"    Specificity: {test_metrics['specificity']:.4f}")
print(f"    ROC-AUC: {test_metrics['roc_auc']:.4f}")
print(f"    QWK: {test_metrics['qwk']:.4f}")

print(f"\n📁 Output Files:")
print(f"  Best Model: {model_dir / 'best_cvitsnet_model.pth'}")
print(f"  Final Model: {model_dir / 'cvitsnet_aptos2019.pth'}")
print(f"  Plots: {plots_dir}/")
print(f"  QWK Curve: {plots_dir / 'qwk_curve.png'}")
print(f"  Metrics: {logs_dir / 'training_history.json'}")
print(f"  Best Metrics: {logs_dir / 'best_model_metrics.json'}")

print(f"\n✅ Training completed successfully!")
print("=" * 80)


TRAINING PIPELINE COMPLETED SUCCESSFULLY!

📊 Results Summary:

  Final Training Metrics (Epoch 100):
    Loss: 0.4441
    Accuracy: 0.8405
    F1 Score: 0.8379
    ROC-AUC: 0.9691
    QWK: 0.8501

  Final Validation Metrics (Epoch 100):
    Loss: 1.3001
    Accuracy: 0.7006
    F1 Score: 0.6758
    ROC-AUC: 0.8859
    QWK: 0.6604

  Best Model: Epoch 91 (val_QWK: 0.7313)

  Test Set Metrics (Best Model):
    Loss: 0.9972
    Accuracy: 0.7249
    Precision: 0.7217
    Recall: 0.7249
    F1 Score: 0.6929
    Specificity: 0.9242
    ROC-AUC: 0.9053
    QWK: 0.7146

📁 Output Files:
  Best Model: trained_model\best_cvitsnet_model.pth
  Final Model: trained_model\cvitsnet_aptos2019.pth
  Plots: results\Plots/
  QWK Curve: results\Plots\qwk_curve.png
  Metrics: results\logs\training_history.json
  Best Metrics: results\logs\best_model_metrics.json

✅ Training completed successfully!


In [11]:
# ============================================================
# Best Model Analysis
# ============================================================

# Find best epoch based on validation QWK (primary metric for APTOS2019)
best_epoch_qwk = int(np.argmax(history['qwk']['val'])) + 1
best_epoch_loss = int(np.argmin(history['loss']['val'])) + 1
best_epoch_acc = int(np.argmax(history['accuracy']['val'])) + 1

print("="*80)
print("BEST MODEL ANALYSIS")
print("="*80)

# Best by QWK
idx = best_epoch_qwk - 1
print(f"\nBest Epoch by Validation QWK: Epoch {best_epoch_qwk}")
print(f"  Train Loss: {history['loss']['train'][idx]:.4f} | Val Loss: {history['loss']['val'][idx]:.4f}")
print(f"  Train Acc:  {history['accuracy']['train'][idx]:.4f} | Val Acc:  {history['accuracy']['val'][idx]:.4f}")
print(f"  Train Prec: {history['precision']['train'][idx]:.4f} | Val Prec: {history['precision']['val'][idx]:.4f}")
print(f"  Train Rec:  {history['recall']['train'][idx]:.4f} | Val Rec:  {history['recall']['val'][idx]:.4f}")
print(f"  Train F1:   {history['f1_score']['train'][idx]:.4f} | Val F1:   {history['f1_score']['val'][idx]:.4f}")
print(f"  Train Spec: {history['specificity']['train'][idx]:.4f} | Val Spec: {history['specificity']['val'][idx]:.4f}")
print(f"  Train AUC:  {history['roc_auc']['train'][idx]:.4f} | Val AUC:  {history['roc_auc']['val'][idx]:.4f}")
print(f"  Train QWK:  {history['qwk']['train'][idx]:.4f} | Val QWK:  {history['qwk']['val'][idx]:.4f}")

# Best by Loss
idx2 = best_epoch_loss - 1
print(f"\nBest Epoch by Validation Loss: Epoch {best_epoch_loss}")
print(f"  Val Loss: {history['loss']['val'][idx2]:.4f} | Val Acc: {history['accuracy']['val'][idx2]:.4f} | Val QWK: {history['qwk']['val'][idx2]:.4f}")

# Best by Accuracy
idx3 = best_epoch_acc - 1
print(f"\nBest Epoch by Validation Accuracy: Epoch {best_epoch_acc}")
print(f"  Val Loss: {history['loss']['val'][idx3]:.4f} | Val Acc: {history['accuracy']['val'][idx3]:.4f} | Val QWK: {history['qwk']['val'][idx3]:.4f}")

print(f"\nCurrently saved model: Epoch 100 (final epoch)")
print("="*80)

BEST MODEL ANALYSIS

Best Epoch by Validation QWK: Epoch 91
  Train Loss: 0.5076 | Val Loss: 1.1525
  Train Acc:  0.8006 | Val Acc:  0.7238
  Train Prec: 0.7941 | Val Prec: 0.7016
  Train Rec:  0.8006 | Val Rec:  0.7238
  Train F1:   0.7929 | Val F1:   0.6852
  Train Spec: 0.9472 | Val Spec: 0.9228
  Train AUC:  0.9586 | Val AUC:  0.8859
  Train QWK:  0.7945 | Val QWK:  0.7313

Best Epoch by Validation Loss: Epoch 31
  Val Loss: 0.7584 | Val Acc: 0.7355 | Val QWK: 0.6187

Best Epoch by Validation Accuracy: Epoch 67
  Val Loss: 0.8953 | Val Acc: 0.7529 | Val QWK: 0.6915

Currently saved model: Epoch 100 (final epoch)
